In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load environment variables first
load_dotenv()
parser = StrOutputParser()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
URL = os.getenv("URL")
API_KEY = os.getenv("APIKEY")

# 2. Initialize Models
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0,api_key=GROQ_API_KEY)




In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
# 3. File Loading
txtfile_path = "swatvalley.txt"
loader = TextLoader(txtfile_path)
txtfile = loader.load()

# 4. Text Splitting
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",   # paragraphs (highest priority)
        "\n",     # lines
        ". ",     # sentences
        " ",      # words
        ""        # fallback
    ]
)
split_document = splitter.split_documents(txtfile)

# 5. Vector Store Ingestion
vectorstore = QdrantVectorStore.from_documents(
    documents=split_document,
    embedding=embeddings,
    api_key=API_KEY,
    url=URL,
    collection_name="vanilla_rag",
)

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

rag_prompt = ChatPromptTemplate.from_template(
    """"
    Please answer the following questions,if it is in context otherwise answer "I don,t have enough information about this"
    Context:
    {context}

    Question:
    {question}


    """
)

In [8]:
#This return objects in a string format
def get_content(docs):
    return "\n\n".join( doc.page_content for doc in docs )

In [9]:
#context and question will fill this(rag_prompt)
#LCEL

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser)

In [11]:
question = "Where is malam jabba in swat"
answer = rag_chain.invoke(question)

print(answer)

Malam Jabba is located about 44 km from Mingora in Swat.


In [15]:
question2 = "Where is bahria university"
answer2 = rag_chain.invoke(question2)

print(answer2)

I don't have enough information about this.


In [18]:
question = "Where is malam jabba in swat"
answer = rag_chain.stream(question)
for chunk in answer:
    print(chunk, end="", flush=True)

Malam Jabba is located about 44 km from Mingora in Swat.